### This file compares results of Lexical, Semantic and Hybrid search across various models

In [2]:
import pandas as pd
df_mr = pd.read_csv('.././datasets/translation/pure_marathi.csv')
print(df_mr.shape)
df_mr.head()

(64, 5)


,scheme_id,site,scheme_name,description,scheme_link
0,1,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,आनंदाचा शिधा,दि. 04.10.2022 च्या शासन निर्णयानुसार राष्ट्री...,https://mahafood.gov.in/scheme/%e0%a4%86%e0%a4...
1,2,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,एपीएल शेतकरी,"राज्यातील छत्रपती संभाजीनगर, जालना, नांदेड, बी...",https://mahafood.gov.in/scheme/%e0%a4%8f%e0%a4...
2,3,https://mahafood.gov.in/provider/%E0%A4%B0%E0%...,शिवभोजन,राज्यातील गरीब व गरजू जनतेला सवलतीच्या दरात भो...,https://mahafood.gov.in/scheme/%e0%a4%b6%e0%a4...
3,4,https://maharashtra.gov.in/Site/1604/scheme,कृषी योजना,शेतकरी वर्गासाठी राज्य शासनातर्फे अनेक योजना उ...,https://www.manage.gov.in/fpoacademy/SGSchemes...
4,5,https://maharashtra.gov.in/Site/1604/scheme,कृषी तारण कर्ज योजना,शेतकऱ्याला असलेल्या आर्थिक गरजेपोटी तसेच स्थान...,https://www.msamb.com/Schemes/PledgeFinance


In [1]:
df_en = pd.read_csv('.././datasets/translation/pure_english.csv')
print(df_en.shape)
df_en.head()

NameError: name 'pd' is not defined

In [3]:
import numpy as np
print(np.__version__)


2.3.3


In [4]:
from sentence_transformers import SentenceTransformer
mahaSBERT_model = SentenceTransformer('l3cube-pune/marathi-sentence-similarity-sbert')

In [5]:
from sentence_transformers import SentenceTransformer
indicSBERT_model = SentenceTransformer('l3cube-pune/indic-sentence-similarity-sbert')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--l3cube-pune--indic-sentence-similarity-sbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
import pandas as pd
import numpy as np
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
import math


In [17]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200",
                   basic_auth = ('elasticsearch', 'e3TKzHmKRFWBP4gY--cjeQ'),
                   request_timeout=60,
                   )
es.ping() 
print(es.info())

{'name': 'LAPTOP-ANN0J427', 'cluster_name': 'elasticsearch', 'cluster_uuid': '6AxnonBCTU-8fGsKa_EYMQ', 'version': {'number': '9.1.3', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '0c781091a2f57de895a73a1391ff8426c0153c8d', 'build_date': '2025-08-24T22:05:04.526302670Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [ ]:
from src.elasticSearch.indexMappings import indexMappings
es.indices.create(index = "pure_marathi" , mappings = indexMappings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': False, 'index': 'schemes_mapping'})

In [21]:
df_mr["mahasbert_des_vector"] = df_mr["description"].apply(lambda x: mahaSBERT_model.encode(x))

In [22]:
marathi_record_list = df_mr.to_dict("records")

In [ ]:
marathi_record_list[0]

In [24]:
for record in marathi_record_list:
    try:
        es.index(index="pure_marathi", document=record, id=record['scheme_id'])
    except Exception as e:
        print("error", e)

In [25]:
es.count(index="pure_marathi")

ObjectApiResponse({'count': 64, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [8]:
def lexical_search(index_name, query_text, k=10):
    body = {
        "size": k,
        "query": {
            "multi_match": {
                "query": query_text,
                "fields": ["scheme_name", "description"]
            }
        }
    }
    res = es.search(index=index_name, body=body)
    return [int(hit["_source"]["scheme_id"]) for hit in res["hits"]["hits"]]


In [9]:
def semantic_search(index_name, query_text, model, k=10):
    query_vec = model.encode(query_text).tolist()

    body = {
        "size": k,
        "query": {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.query_vector, 'embedding')",
                    "params": {"query_vector": query_vec}
                }
            }
        }
    }

    res = es.search(index=index_name, body=body)
    return [int(hit["_source"]["scheme_id"]) for hit in res["hits"]["hits"]]


In [10]:
def hybrid_search(index_name, query_text, model, k=10, alpha=0.5):
    query_vec = model.encode(query_text).tolist()

    body = {
        "size": k,
        "query": {
            "bool": {
                "should": [
                    {
                        "multi_match": {
                            "query": query_text,
                            "fields": ["scheme_name^2", "description"]
                        }
                    },
                    {
                        "script_score": {
                            "query": {"match_all": {}},
                            "script": {
                                "source": """
                                double bm25 = _score;
                                double vec = cosineSimilarity(params.qv, 'embedding');
                                return (bm25 * (1 - params.alpha)) + (vec * params.alpha);
                                """,
                                "params": {
                                    "qv": query_vec,
                                    "alpha": alpha
                                }
                            }
                        }
                    }
                ]
            }
        }
    }

    res = es.search(index=index_name, body=body)
    return [int(hit["_source"]["scheme_id"]) for hit in res["hits"]["hits"]]


In [11]:
def precision_at_k(relevant, retrieved, k=10):
    retrieved = retrieved[:k]
    rel_set = set(relevant)
    return sum([1 for r in retrieved if r in rel_set]) / k


def recall_at_k(relevant, retrieved, k=10):
    rel_set = set(relevant)
    retrieved = retrieved[:k]
    return sum([1 for r in retrieved if r in rel_set]) / len(rel_set)


def mrr(relevant, retrieved):
    for idx, r in enumerate(retrieved, start=1):
        if r in relevant:
            return 1 / idx
    return 0


def ndcg_at_k(relevant, retrieved, k=10):
    def dcg(rel):
        return sum([1 / math.log2(i+2) for i in range(len(rel))])
    
    rel = [1 if r in relevant else 0 for r in retrieved[:k]]
    ideal = sorted(rel, reverse=True)
    return dcg(rel) / (dcg(ideal) or 1)


In [12]:
test_marathi = pd.read_csv("testSets/marathi.csv")
test_english = pd.read_csv("testSets/english.csv")

print("Test sets loaded ✅")


Test sets loaded ✅


In [13]:
def evaluate(test_df, index_name, language):
    results = []

    for _, row in test_df.iterrows():
        query = row["query"]
        relevant = list(map(int, row["relevant_scheme_ids"].split(",")))

        for model_name, model in [("mahaSBERT", mahaSBERT_model), ("indicSBERT", indicSBERT_model)]:

            # Run searches
            lex_res = lexical_search(index_name, query)
            sem_res = semantic_search(index_name, query, model)
            hyb_res = hybrid_search(index_name, query, model)

            for search_type, retrieved in [
                ("lexical", lex_res),
                ("semantic", sem_res),
                ("hybrid", hyb_res)
            ]:
                results.append({
                    "language": language,
                    "model": model_name,
                    "search_type": search_type,
                    "precision@10": precision_at_k(relevant, retrieved),
                    "recall@10": recall_at_k(relevant, retrieved),
                    "mrr": mrr(relevant, retrieved),
                    "ndcg@10": ndcg_at_k(relevant, retrieved)
                })

    return pd.DataFrame(results)


In [ ]:
marathi_results = evaluate(test_marathi, "marathi_schemes", "marathi")
english_results = evaluate(test_english, "english_schemes", "english")

final_results = pd.concat([marathi_results, english_results])

final_results
